In [23]:
#Insert a single chain .pdb file here:

pdb_filepath=''

#Multi-chain .pdb files will be considered as a single protein!

#The script will generate a .csv file containing the HyBind value measured around any superficial residue.
#Insert here the desired filepath of the .csv file:

csv_filepath=''

In [24]:
import os
from Bio import PDB
import numpy as np
import pandas as pd
from Bio.PDB.SASA import ShrakeRupley
from tqdm import tqdm
import io
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
import copy




#ShrakeRupley computes the SASA
sr = ShrakeRupley()

parser = PDB.PDBParser(QUIET=True)

if pdb_filepath.endswith('.pdb'):

      structure = parser.get_structure('protein', pdb_filepath)
      sr.compute(structure, level='R')

      data = []

      for model in structure:
        for chain in model:
          for residue in chain:
            if 'CA' in residue:

                atom = residue['CA']

                data.append({
                        'File': pdb_filepath,
                        'Chain': 'A',
                        'Residue': residue.get_resname(),
                        'Residue ID': residue.get_id(),
                        'Coordinates': atom.get_coord(),
                        'SASA': structure[model.id][chain.id][residue.get_id()].sasa
                })


      prot = pd.DataFrame(data)


      coord=prot['Coordinates']
      residuo=prot['Residue']
      proteina=prot['Chain']
      sasa=prot['SASA']


else:

    print('Error: the supplied file has not .pdb extension')


#creating the output dataframe

df={
    'Index': np.arange(1,len(coord)+1),
    'Residue': residuo,
    'SASA': sasa,
    
}


csv_output = pd.DataFrame(df)

#PCA


def PCA(points, c):

  sigma=np.zeros((3,3))
  centro=np.zeros(3)

  for k in range(len(points)):
    centro[0]=centro[0]+points[k][0]
    centro[1]=centro[1]+points[k][1]
    centro[2]=centro[2]+points[k][2]

  centro[0]=centro[0]/len(points)
  centro[1]=centro[1]/len(points)
  centro[2]=centro[2]/len(points)

  for i in range(3):
    for j in range(3):
      for k in range(len(points)):
        sigma[i, j]=sigma[i, j]+(points[k][i]-centro[i])*(points[k][j]-centro[j])



  eigenvalues, eigenvectors = np.linalg.eig(sigma)



  a=int(0)

  for i in range(3):

    if eigenvalues[i]<eigenvalues[a]:
      a=i



  axes=np.delete(eigenvectors, a, axis=1)

  return axes


#Amino acid names
name=['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']
    
#Hydrophobicity scale
hydval=[0.8584337349397589, 0.5180722891566265, 0.3885542168674699, 0.03614457831325291, 0.822289156626506, 0.6204819277108433, 0.0, 0.7590361445783131, 0.31024096385542166, 1.0, 0.7620481927710843, 0.16867469879518074, 0.8855421686746988, 0.9277108433734941, 0.8343373493975903, 0.5602409638554217, 0.8012048192771084, 0.8042168674698795, 0.75, 0.8614457831325302]


#Average BS and RP g(r) distributions and corresponding standard deviations
gBS=np.array([0.6305885913557024, 0.6165946744293109, 0.5967685822091394, 0.5825924042034424, 0.5605124055293239, 0.5489381058830032, 0.5335332943673685, 0.5220270860451791, 0.5168725608977874, 0.5107359567360685])
sigmaBS=np.array([0.2693558308454179, 0.21307976030095843, 0.17511219913030981, 0.1518287813533818, 0.13856658725903134, 0.12861292393447377, 0.12585837192726632, 0.12727860276929975, 0.14385229141525466, 0.18505653065541933])

gRP=np.array([0.5050844478390494, 0.5274388018050407, 0.5200275233952031, 0.5236423104604931, 0.5148282389885863, 0.5090821102398427, 0.5071068497699421, 0.5073982717793842, 0.50660427065554, 0.528694739730556])
sigmaRP=np.array([0.2895509884974675, 0.22796282533196685, 0.2040648608317668, 0.2027533757914061, 0.1858352070009627, 0.18637184704797102, 0.19143401635929572, 0.20102279996330685, 0.22196137939877889, 0.2566718289167629])

nbins=10

#Measuring the HBind values with 10 bins and r_MAX=21A

HyBind=np.full(len(coord), np.nan)

for i in range(len(coord)):
    

    if sasa[i]>0:

        cen=coord[i]

        pos=np.array([coord[j] for j in range(len(coord)) if np.linalg.norm(coord[j]-coord[i])<21 and sasa[j]>0])
        res=np.array([residuo[j] for j in range(len(coord)) if np.linalg.norm(coord[j]-coord[i])<21 and sasa[j]>0])
        sup=np.array([sasa[j] for j in range(len(coord)) if np.linalg.norm(coord[j]-coord[i])<21 and sasa[j]>0])


        hyd=[]
        

        for nn in res:

            for l in range(20):

                if nn == name[l]:

                    hyd.append(hydval[l])   

                    

        hyd=np.array(hyd)    

            

        

        aa=PCA(pos, cen)

        Hr=np.zeros(nbins)
        norm=np.zeros(nbins)


        for k in range(len(pos)):

            dist=np.sqrt((np.dot(pos[k]-cen, aa[:, 0]))**2+(np.dot(pos[k]-cen, aa[:, 1])**2))    
    
            for l in range(nbins):
    
                if (dist > l*2.1 and dist <= (l+1)*2.1) or (dist==0 and l==0):
                    Hr[l]=Hr[l]+hyd[k]*sasa[k]
                    norm[l]=norm[l]+sasa[k]


        gr=np.zeros(nbins)

        for l in range(nbins):

            if norm[l]>0:
                gr[l]=Hr[l]/norm[l]

            else:
                gr[l]=100
    
    

        
        chi2=0
        y=0    
        n=0
    
        for l in range(nbins):
    
          if norm[l]>0:
    
              chi2=chi2+(gr[l]-gBS[l])**2/sigmaBS[l]**2
              y=y+(gr[l]-gRP[l])**2/sigmaRP[l]**2
    
              n=n+1



        HyBind[i]=(chi2-y)/n
    
    



csv_output.insert(3, 'HyBind', HyBind)
                             
csv_output.to_csv(csv_filepath, index=False)
